# 01 Process HD/PD clinical and cost data

This notebook cleans and validates the `data_raw` inputs for the Singapore haemodialysis (HD) versus peritoneal dialysis (PD) health economics portfolio project.

This notebook creates the `data_processed` layer that the modelling notebook will use.

Main tasks:

1. Load all raw CSVs.
2. Check that the source labels, numeric values and proportions are internally coherent.
3. Convert raw source tables into model-ready input tables.
4. Create a base-case parameter table.
5. Create a Markov transition matrix for the first model.
6. Create budget impact scenario inputs.
7. Save validation outputs so the report can transparently explain what is strong, weak and assumption-driven.

Important: several inputs are cost proxies, assumptions or literature-derived placeholders. This notebook keeps those labels visible rather than hiding uncertainty.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
from datetime import datetime

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# Detect project root whether this notebook is run from:
# 1. HD_PD_HTA_PROJECT/
# 2. HD_PD_HTA_PROJECT/notebooks/
cwd = Path.cwd()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_RAW = PROJECT_ROOT / "data_raw"
DATA_PROCESSED = PROJECT_ROOT / "data_processed"
OUTPUTS = PROJECT_ROOT / "outputs"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_RAW)
print("Processed data folder:", DATA_PROCESSED)

Project root: /Users/marissa/Desktop/portfolio/HD_PD_HTA_PROJECT
Raw data folder: /Users/marissa/Desktop/portfolio/HD_PD_HTA_PROJECT/data_raw
Processed data folder: /Users/marissa/Desktop/portfolio/HD_PD_HTA_PROJECT/data_processed


## 1. Load raw CSV files

The notebook expects the nine raw CSV files generated from `00_generate_hd_pd_data_raw.ipynb`.

If any file is missing, stop here and fix the `data_raw` folder before processing.

In [2]:
RAW_FILES = {
    "model_scope": "model_scope_raw.csv",
    "population": "population_inputs_raw.csv",
    "modality_mix": "modality_mix_raw.csv",
    "clinical_evidence": "clinical_outcomes_evidence_raw.csv",
    "costs": "cost_inputs_raw.csv",
    "utilities": "utility_inputs_raw.csv",
    "transitions": "transition_inputs_raw.csv",
    "uptake": "uptake_scenarios_raw.csv",
    "assumptions": "assumptions_register_raw.csv",
}

missing_files = [name for name, filename in RAW_FILES.items() if not (DATA_RAW / filename).exists()]
if missing_files:
    raise FileNotFoundError(f"Missing raw CSV files: {missing_files}")

raw = {}
for name, filename in RAW_FILES.items():
    path = DATA_RAW / filename
    raw[name] = pd.read_csv(path)
    print(f"{name:18} {filename:42} rows={len(raw[name])}, cols={len(raw[name].columns)}")

model_scope        model_scope_raw.csv                        rows=16, cols=9
population         population_inputs_raw.csv                  rows=5, cols=12
modality_mix       modality_mix_raw.csv                       rows=6, cols=12
clinical_evidence  clinical_outcomes_evidence_raw.csv         rows=5, cols=14
costs              cost_inputs_raw.csv                        rows=11, cols=13
utilities          utility_inputs_raw.csv                     rows=7, cols=13
transitions        transition_inputs_raw.csv                  rows=6, cols=14
uptake             uptake_scenarios_raw.csv                   rows=20, cols=12
assumptions        assumptions_register_raw.csv               rows=8, cols=13


## 2. Helper functions

These functions keep the processing reproducible and reduce the chance of silent data errors.

In [3]:
def clean_colnames(df: pd.DataFrame) -> pd.DataFrame:
    """Standardise column names to lower snake_case."""
    out = df.copy()
    out.columns = [
        re.sub(r"[^0-9a-zA-Z]+", "_", c.strip().lower()).strip("_")
        for c in out.columns
    ]
    return out

def to_bool_series(s: pd.Series) -> pd.Series:
    """Convert common text/bool representations to Boolean where possible."""
    return (
        s.astype(str)
         .str.strip()
         .str.lower()
         .map({"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False})
    )

def numeric_or_nan(x):
    """Convert numeric-like values to float, otherwise return NaN."""
    try:
        if pd.isna(x):
            return np.nan
        return float(str(x).replace(",", "").strip())
    except Exception:
        return np.nan

def get_param(df: pd.DataFrame, parameter: str, value_col: str = "base_value"):
    """Return one parameter value from a table with a parameter column."""
    match = df.loc[df["parameter"] == parameter, value_col]
    if match.empty:
        raise KeyError(f"Parameter not found: {parameter}")
    return match.iloc[0]

def add_validation(results: list, test: str, status: str, details: str, severity: str = "check"):
    results.append({
        "test": test,
        "status": status,
        "details": details,
        "severity": severity,
        "checked_at": datetime.now().isoformat(timespec="seconds")
    })

def write_processed(df: pd.DataFrame, filename: str):
    path = DATA_PROCESSED / filename
    df.to_csv(path, index=False)
    print(f"Wrote {filename}: {len(df)} rows")
    return path

## 3. Standardise raw tables

This step does light cleaning only. It should not change the meaning of the raw source data.

In [4]:
tables = {name: clean_colnames(df) for name, df in raw.items()}

# Convert known numeric fields where present.
for name, df in tables.items():
    for col in ["base_value", "low_value", "high_value", "hd_value", "pd_value",
                "year", "incident_patients_per_year", "pd_suitable_share",
                "eligible_patients_per_year", "pd_uptake", "hd_uptake"]:
        if col in df.columns:
            df[col] = df[col].apply(numeric_or_nan)

# Convert include_in_base_case columns to Boolean.
for name in ["costs", "utilities", "transitions"]:
    if "include_in_base_case" in tables[name].columns:
        tables[name]["include_in_base_case"] = to_bool_series(tables[name]["include_in_base_case"])

# Save a copy of lightly cleaned raw tables as processed mirrors.
for name, df in tables.items():
    write_processed(df, f"{name}_cleaned.csv")

tables["costs"].head()

Wrote model_scope_cleaned.csv: 16 rows
Wrote population_cleaned.csv: 5 rows
Wrote modality_mix_cleaned.csv: 6 rows
Wrote clinical_evidence_cleaned.csv: 5 rows
Wrote costs_cleaned.csv: 11 rows
Wrote utilities_cleaned.csv: 7 rows
Wrote transitions_cleaned.csv: 6 rows
Wrote uptake_cleaned.csv: 20 rows
Wrote assumptions_cleaned.csv: 8 rows


,parameter,modality,base_value,low_value,high_value,unit,qualifier,include_in_base_case,notes,source_key,source_title,source_url,data_origin
0,hd_monthly_cost,HD,3150.0,2800.0,3500.0,SGD_per_month,midpoint_of_range,True,"Patient-facing monthly HD cost range, excludin...",DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Singapore patient-facing HD and PD cost ranges
1,hd_annual_cost,HD,37800.0,33600.0,42000.0,SGD_per_year,derived_from_monthly,True,Derived from monthly HD cost.,DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Singapore patient-facing HD and PD cost ranges
2,capd_monthly_cost,CAPD,1200.0,1100.0,1300.0,SGD_per_month,midpoint_of_range,False,Patient-facing CAPD cost range from HealthHub.,HEALTHHUB_PD_COSTS,HealthHub Singapore. Peritoneal Dialysis,https://www.healthhub.sg/health-conditions/wha...,Singapore patient-facing PD cost information
3,capd_annual_cost,CAPD,14400.0,13200.0,15600.0,SGD_per_year,derived_from_monthly,False,Derived from monthly CAPD cost.,HEALTHHUB_PD_COSTS,HealthHub Singapore. Peritoneal Dialysis,https://www.healthhub.sg/health-conditions/wha...,Singapore patient-facing PD cost information
4,apd_monthly_cost,APD,1700.0,1600.0,1800.0,SGD_per_month,midpoint_of_range,False,"Patient-facing APD cost range, excluding elect...",HEALTHHUB_PD_COSTS,HealthHub Singapore. Peritoneal Dialysis,https://www.healthhub.sg/health-conditions/wha...,Singapore patient-facing PD cost information


## 4. Validate data structure and source hygiene

This section checks the things that would otherwise quietly break the model:

- required columns,
- old source labels,
- proportions that should sum to 1,
- cost derivations,
- population derivations,
- annual uptake scenario consistency.

In [5]:
validation = []

required_columns = {
    "model_scope": ["section", "field", "value", "unit", "notes", "source_key", "source_title", "source_url", "data_origin"],
    "population": ["parameter", "base_value", "low_value", "high_value", "unit", "qualifier", "use_in_model", "notes", "source_key", "source_title", "source_url", "data_origin"],
    "modality_mix": ["parameter", "base_value", "low_value", "high_value", "unit", "qualifier", "use_in_model", "notes", "source_key", "source_title", "source_url", "data_origin"],
    "costs": ["parameter", "modality", "base_value", "low_value", "high_value", "unit", "qualifier", "include_in_base_case", "notes", "source_key", "source_title", "source_url", "data_origin"],
    "utilities": ["parameter", "modality", "base_value", "low_value", "high_value", "unit", "qualifier", "include_in_base_case", "notes", "source_key", "source_title", "source_url", "data_origin"],
    "transitions": ["parameter", "from_state", "to_state", "base_value", "low_value", "high_value", "unit", "qualifier", "include_in_base_case", "notes", "source_key", "source_title", "source_url", "data_origin"],
    "uptake": ["year", "scenario", "incident_patients_per_year", "pd_suitable_share", "eligible_patients_per_year", "pd_uptake", "hd_uptake", "notes", "source_key", "source_title", "source_url", "data_origin"],
    "assumptions": ["assumption_id", "assumption", "base_value", "low_value", "high_value", "uncertainty_type", "validation_needed", "model_impact", "notes", "source_key", "source_title", "source_url", "data_origin"],
}

for name, cols in required_columns.items():
    missing = [c for c in cols if c not in tables[name].columns]
    if missing:
        add_validation(validation, f"{name}: required columns", "FAIL", f"Missing columns: {missing}", "critical")
    else:
        add_validation(validation, f"{name}: required columns", "PASS", "All required columns present.", "critical")

# Source hygiene check.
all_text = "\n".join(
    tables[name].astype(str).agg(" ".join, axis=1).str.cat(sep="\n")
    for name in tables
)

banned_terms = ["ACE_METHODS", "Agency for Care Effectiveness. Medical Technology Evaluation Methods and Process Guide / Process and Methods"]
found_banned = [term for term in banned_terms if term in all_text]
if found_banned:
    add_validation(validation, "Source hygiene: old source labels", "FAIL", f"Old labels found: {found_banned}", "critical")
else:
    add_validation(validation, "Source hygiene: old source labels", "PASS", "No old ACE_METHODS or old long source title found.", "critical")

# Duplicate parameter checks in parameter tables.
for name in ["population", "modality_mix", "costs", "utilities", "transitions"]:
    df = tables[name]
    if "parameter" in df.columns:
        duplicated = sorted(df.loc[df["parameter"].duplicated(), "parameter"].unique())
        if duplicated:
            add_validation(validation, f"{name}: duplicate parameters", "WARN", f"Duplicated parameters: {duplicated}", "data_quality")
        else:
            add_validation(validation, f"{name}: duplicate parameters", "PASS", "No duplicate parameter names.", "data_quality")

validation_df = pd.DataFrame(validation)
validation_df

,test,status,details,severity,checked_at
0,model_scope: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
1,population: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
2,modality_mix: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
3,costs: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
4,utilities: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
5,transitions: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
6,uptake: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
7,assumptions: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
8,Source hygiene: old source labels,PASS,No old ACE_METHODS or old long source title fo...,critical,2026-05-06T21:36:01
9,population: duplicate parameters,PASS,No duplicate parameter names.,data_quality,2026-05-06T21:36:01


In [6]:
# Numerical consistency checks

# 1. Cost derivation checks.
costs = tables["costs"].copy()
hd_monthly = get_param(costs, "hd_monthly_cost")
hd_annual = get_param(costs, "hd_annual_cost")
capd_monthly = get_param(costs, "capd_monthly_cost")
capd_annual = get_param(costs, "capd_annual_cost")
apd_monthly = get_param(costs, "apd_monthly_cost")
apd_annual = get_param(costs, "apd_annual_cost")
pd_blended_monthly = get_param(costs, "pd_blended_monthly_cost")
pd_blended_annual = get_param(costs, "pd_blended_annual_cost")

checks = [
    ("HD annual = monthly × 12", hd_annual, hd_monthly * 12),
    ("CAPD annual = monthly × 12", capd_annual, capd_monthly * 12),
    ("APD annual = monthly × 12", apd_annual, apd_monthly * 12),
    ("Blended PD annual = monthly × 12", pd_blended_annual, pd_blended_monthly * 12),
]

for label, actual, expected in checks:
    if np.isclose(actual, expected, atol=0.01):
        add_validation(validation, label, "PASS", f"Actual={actual:.2f}; expected={expected:.2f}.", "calculation")
    else:
        add_validation(validation, label, "FAIL", f"Actual={actual:.2f}; expected={expected:.2f}.", "critical")

# 2. APD/CAPD share check.
mix = tables["modality_mix"].copy()
apd_share = get_param(mix, "apd_share_of_pd_patients")
capd_share = get_param(mix, "capd_share_of_pd_patients")
share_sum = apd_share + capd_share

if np.isclose(share_sum, 1.0, atol=0.0001):
    add_validation(validation, "APD + CAPD shares", "PASS", f"Base shares sum to {share_sum:.4f}.", "calculation")
else:
    add_validation(validation, "APD + CAPD shares", "FAIL", f"Base shares sum to {share_sum:.4f}, not 1.", "critical")

expected_pd_blended = apd_share * apd_monthly + capd_share * capd_monthly
if np.isclose(pd_blended_monthly, expected_pd_blended, atol=0.01):
    add_validation(validation, "Blended PD monthly cost", "PASS", f"Actual={pd_blended_monthly:.2f}; expected={expected_pd_blended:.2f}.", "calculation")
else:
    add_validation(validation, "Blended PD monthly cost", "FAIL", f"Actual={pd_blended_monthly:.2f}; expected={expected_pd_blended:.2f}.", "critical")

# 3. Population derivation checks.
pop = tables["population"].copy()
new_per_day = get_param(pop, "new_kidney_failure_patients_per_day")
new_per_year = get_param(pop, "estimated_new_kidney_failure_patients_per_year")
expected_per_year = new_per_day * 365

if np.isclose(new_per_year, expected_per_year, atol=0.01):
    add_validation(validation, "Incident patients per year", "PASS", f"Actual={new_per_year:.0f}; expected={expected_per_year:.0f}.", "calculation")
else:
    add_validation(validation, "Incident patients per year", "FAIL", f"Actual={new_per_year:.0f}; expected={expected_per_year:.0f}.", "critical")

# 4. Uptake scenario checks.
uptake = tables["uptake"].copy()
uptake["share_sum"] = uptake["pd_uptake"] + uptake["hd_uptake"]
bad_uptake = uptake.loc[~np.isclose(uptake["share_sum"], 1.0, atol=0.0001)]

if bad_uptake.empty:
    add_validation(validation, "Uptake shares", "PASS", "PD uptake + HD uptake = 1 for every row.", "calculation")
else:
    add_validation(validation, "Uptake shares", "FAIL", f"{len(bad_uptake)} rows do not sum to 1.", "critical")

uptake["eligible_expected"] = (uptake["incident_patients_per_year"] * uptake["pd_suitable_share"]).round(0)
bad_eligible = uptake.loc[~np.isclose(uptake["eligible_patients_per_year"], uptake["eligible_expected"], atol=1)]

if bad_eligible.empty:
    add_validation(validation, "Eligible patients per year", "PASS", "Eligible patients = incident patients × PD suitability share for every row.", "calculation")
else:
    add_validation(validation, "Eligible patients per year", "FAIL", f"{len(bad_eligible)} rows have inconsistent eligible patient counts.", "critical")

# Update validation table.
validation_df = pd.DataFrame(validation)
write_processed(validation_df, "data_validation_summary.csv")
validation_df

Wrote data_validation_summary.csv: 23 rows


,test,status,details,severity,checked_at
0,model_scope: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
1,population: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
2,modality_mix: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
3,costs: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
4,utilities: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
5,transitions: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
6,uptake: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
7,assumptions: required columns,PASS,All required columns present.,critical,2026-05-06T21:36:01
8,Source hygiene: old source labels,PASS,No old ACE_METHODS or old long source title fo...,critical,2026-05-06T21:36:01
9,population: duplicate parameters,PASS,No duplicate parameter names.,data_quality,2026-05-06T21:36:01


## 5. Create source inventory

This table is useful later for the methods section and reference tracking. It shows every source used in the raw data layer and where it appears.

In [7]:
source_records = []

for table_name, df in tables.items():
    if "source_key" not in df.columns:
        continue

    group_cols = ["source_key", "source_title", "source_url", "data_origin"]
    for _, row in df[group_cols].drop_duplicates().iterrows():
        source_records.append({
            "table": table_name,
            "source_key": row["source_key"],
            "source_title": row["source_title"],
            "source_url": row["source_url"],
            "data_origin": row["data_origin"]
        })

source_inventory = (
    pd.DataFrame(source_records)
    .drop_duplicates()
    .sort_values(["source_key", "table"])
    .reset_index(drop=True)
)

write_processed(source_inventory, "source_inventory.csv")
source_inventory

Wrote source_inventory.csv: 28 rows


,table,source_key,source_title,source_url,data_origin
0,assumptions,COOPER_2020_UTILITIES,Cooper JT et al. Health related quality of lif...,https://pmc.ncbi.nlm.nih.gov/articles/PMC7507735/,Systematic review of CKD health-state utility ...
1,utilities,COOPER_2020_UTILITIES,Cooper JT et al. Health related quality of lif...,https://pmc.ncbi.nlm.nih.gov/articles/PMC7507735/,Systematic review of CKD health-state utility ...
2,assumptions,DRUMMOND_2015,Drummond MF et al. Methods for the Economic Ev...,NaN,Health economic evaluation textbook
3,model_scope,DRUMMOND_2015,Drummond MF et al. Methods for the Economic Ev...,NaN,Health economic evaluation textbook
4,utilities,DRUMMOND_2015,Drummond MF et al. Methods for the Economic Ev...,NaN,Health economic evaluation textbook
5,assumptions,DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Singapore patient-facing HD and PD cost ranges
6,costs,DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Singapore patient-facing HD and PD cost ranges
7,costs,HEALTHHUB_PD_COSTS,HealthHub Singapore. Peritoneal Dialysis,https://www.healthhub.sg/health-conditions/wha...,Singapore patient-facing PD cost information
8,assumptions,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Singapore health economics discussion of kidne...
9,costs,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Singapore health economics discussion of kidne...


## 6. Process model scope and clinical evidence

These are mainly report-facing tables. They are kept clean and ready to insert into the final report, but they do not directly drive the Markov calculations.

In [8]:
model_scope_processed = tables["model_scope"].copy()
model_scope_processed["include_in_report"] = True
model_scope_processed["processed_note"] = "Scope item retained for report methods and decision problem table."
write_processed(model_scope_processed, "model_scope_processed.csv")

clinical_evidence_processed = tables["clinical_evidence"].copy()
clinical_evidence_processed["include_in_report"] = True
clinical_evidence_processed["direct_model_input"] = clinical_evidence_processed["use_in_model"].astype(str).str.contains(
    "Mortality|QALY|transition|probability", case=False, regex=True
)
clinical_evidence_processed["evidence_caution"] = np.where(
    clinical_evidence_processed["evidence_type"].astype(str).str.contains("cohort|observational|retrospective", case=False, regex=True),
    "Observational evidence; interpret modality comparisons cautiously due to selection and case-mix.",
    "Contextual or model evidence; use according to notes."
)
write_processed(clinical_evidence_processed, "clinical_evidence_processed.csv")

model_scope_processed.head()

Wrote model_scope_processed.csv: 16 rows
Wrote clinical_evidence_processed.csv: 5 rows


,section,field,value,unit,notes,source_key,source_title,source_url,data_origin,include_in_report,processed_note
0,project_identity,project_title,Economic Evaluation and Budget Impact Analysis...,text,Report title for portfolio model.,SG_MTE_METHODS,Singapore Medical Technology Evaluation Method...,https://www.ace-hta.gov.sg/resources/process-m...,Singapore medical technology evaluation method...,True,Scope item retained for report methods and dec...
1,decision_problem,decision_problem,Should Singapore increase peritoneal dialysis ...,text,Frames the project as a policy pathway evaluat...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,True,Scope item retained for report methods and dec...
2,population,population,Adults with kidney failure in Singapore who ar...,text,The intervention-eligible population should ex...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,True,Scope item retained for report methods and dec...
3,intervention,intervention,Increased uptake of peritoneal dialysis among ...,text,"Modelled as a modality uptake scenario, not as...",MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,True,Scope item retained for report methods and dec...
4,comparator,comparator,Current practice / haemodialysis-dominant dial...,text,Comparator will be parameterised using modalit...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,True,Scope item retained for report methods and dec...


## 7. Process cost inputs

The modelling notebook needs one clean cost table with only model-ready annual cost parameters.

The base model will use:

- annual HD cost,
- annual blended PD cost,
- annual switched PD-to-HD cost, assumed equal to HD annual cost,
- optional published Yang et al. total cost outputs only as plausibility checks.

The notebook deliberately keeps patient-facing cost proxy notes visible.

In [9]:
costs = tables["costs"].copy()

# Keep all cost rows as a processed audit table.
costs["cost_input_type"] = np.where(
    costs["include_in_base_case"].fillna(False),
    "base_case_cost_input",
    np.where(costs["qualifier"].astype(str).str.contains("published_model_result", case=False), "plausibility_check", "supporting_cost_input")
)
costs["cost_caution"] = np.where(
    costs["notes"].astype(str).str.contains("proxy|patient-facing", case=False, regex=True),
    "Public patient-facing cost proxy; not exact provider cost.",
    "Use according to source and notes."
)
write_processed(costs, "cost_inputs_processed.csv")

# Create model-ready annual costs.
hd_annual_row = costs.loc[costs["parameter"] == "hd_annual_cost"].iloc[0]
pd_annual_row = costs.loc[costs["parameter"] == "pd_blended_annual_cost"].iloc[0]

annual_costs_model_ready = pd.DataFrame([
    {
        "parameter": "annual_cost_hd",
        "health_state": "HD",
        "base_value": hd_annual_row["base_value"],
        "low_value": hd_annual_row["low_value"],
        "high_value": hd_annual_row["high_value"],
        "unit": "SGD_per_year",
        "source_parameter": "hd_annual_cost",
        "source_key": hd_annual_row["source_key"],
        "source_title": hd_annual_row["source_title"],
        "source_url": hd_annual_row["source_url"],
        "notes": "Annual HD cost used in base Markov model."
    },
    {
        "parameter": "annual_cost_pd",
        "health_state": "PD",
        "base_value": pd_annual_row["base_value"],
        "low_value": pd_annual_row["low_value"],
        "high_value": pd_annual_row["high_value"],
        "unit": "SGD_per_year",
        "source_parameter": "pd_blended_annual_cost",
        "source_key": pd_annual_row["source_key"],
        "source_title": pd_annual_row["source_title"],
        "source_url": pd_annual_row["source_url"],
        "notes": "Annual blended PD cost used in base Markov model."
    },
    {
        "parameter": "annual_cost_switched_pd_to_hd",
        "health_state": "Switched_PD_to_HD",
        "base_value": hd_annual_row["base_value"],
        "low_value": hd_annual_row["low_value"],
        "high_value": hd_annual_row["high_value"],
        "unit": "SGD_per_year",
        "source_parameter": "hd_annual_cost",
        "source_key": hd_annual_row["source_key"],
        "source_title": hd_annual_row["source_title"],
        "source_url": hd_annual_row["source_url"],
        "notes": "Assumed equal to annual HD cost after PD technique failure or modality switch."
    },
    {
        "parameter": "annual_cost_death",
        "health_state": "Death",
        "base_value": 0.0,
        "low_value": 0.0,
        "high_value": 0.0,
        "unit": "SGD_per_year",
        "source_parameter": "model_convention",
        "source_key": "DRUMMOND_2015",
        "source_title": "Drummond MF et al. Methods for the Economic Evaluation of Health Care Programmes. 4th ed.",
        "source_url": "NA",
        "notes": "No ongoing dialysis cost assigned after death in this simplified model."
    }
])

write_processed(annual_costs_model_ready, "annual_costs_model_ready.csv")
annual_costs_model_ready

Wrote cost_inputs_processed.csv: 11 rows
Wrote annual_costs_model_ready.csv: 4 rows


,parameter,health_state,base_value,low_value,high_value,unit,source_parameter,source_key,source_title,source_url,notes
0,annual_cost_hd,HD,37800.0,33600.0,42000.0,SGD_per_year,hd_annual_cost,DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Annual HD cost used in base Markov model.
1,annual_cost_pd,PD,18840.0,16800.0,20700.0,SGD_per_year,pd_blended_annual_cost,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Annual blended PD cost used in base Markov model.
2,annual_cost_switched_pd_to_hd,Switched_PD_to_HD,37800.0,33600.0,42000.0,SGD_per_year,hd_annual_cost,DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Assumed equal to annual HD cost after PD techn...
3,annual_cost_death,Death,0.0,0.0,0.0,SGD_per_year,model_convention,DRUMMOND_2015,Drummond MF et al. Methods for the Economic Ev...,NA,No ongoing dialysis cost assigned after death ...


## 8. Process utility inputs

The QALY model uses health-state utility weights. This is one of the weakest data areas, so the processed table labels these inputs as high-uncertainty.

In [10]:
utilities = tables["utilities"].copy()

utilities["utility_input_type"] = np.where(
    utilities["include_in_base_case"].fillna(False),
    "base_case_utility_input",
    np.where(utilities["qualifier"].astype(str).str.contains("published_model_result", case=False), "plausibility_check", "supporting_utility_input")
)

utilities["utility_caution"] = np.where(
    utilities["include_in_base_case"].fillna(False),
    "High-uncertainty utility input; must be tested in sensitivity analysis.",
    "Not directly used in base-case Markov model."
)

write_processed(utilities, "utility_inputs_processed.csv")

utility_map = {
    "health_state_utility_hd": "HD",
    "health_state_utility_pd": "PD",
    "health_state_utility_switched_pd_to_hd": "Switched_PD_to_HD",
    "health_state_utility_death": "Death",
}

utility_rows = []
for param, state in utility_map.items():
    row = utilities.loc[utilities["parameter"] == param].iloc[0]
    utility_rows.append({
        "parameter": param,
        "health_state": state,
        "base_value": row["base_value"],
        "low_value": row["low_value"],
        "high_value": row["high_value"],
        "unit": "utility_weight",
        "source_key": row["source_key"],
        "source_title": row["source_title"],
        "source_url": row["source_url"],
        "notes": row["notes"]
    })

utilities_model_ready = pd.DataFrame(utility_rows)
write_processed(utilities_model_ready, "utilities_model_ready.csv")
utilities_model_ready

Wrote utility_inputs_processed.csv: 7 rows
Wrote utilities_model_ready.csv: 4 rows


,parameter,health_state,base_value,low_value,high_value,unit,source_key,source_title,source_url,notes
0,health_state_utility_hd,HD,0.58,0.44,0.71,utility_weight,COOPER_2020_UTILITIES,Cooper JT et al. Health related quality of lif...,https://pmc.ncbi.nlm.nih.gov/articles/PMC7507735/,Placeholder utility for first-pass model. Coop...
1,health_state_utility_pd,PD,0.62,0.53,0.72,utility_weight,YANG_2018_PD_HRQoL,Yang F et al. Health-Related Quality of Life i...,https://link.springer.com/article/10.1007/s416...,Placeholder utility for first-pass model. Keep...
2,health_state_utility_switched_pd_to_hd,Switched_PD_to_HD,0.56,0.44,0.71,utility_weight,COOPER_2020_UTILITIES,Cooper JT et al. Health related quality of lif...,https://pmc.ncbi.nlm.nih.gov/articles/PMC7507735/,Assumed slightly lower than stable HD due to p...
3,health_state_utility_death,Death,0.00,0.00,0.00,utility_weight,DRUMMOND_2015,Drummond MF et al. Methods for the Economic Ev...,NaN,Death has zero utility in QALY modelling.


## 9. Process transition inputs

The base-case model uses annual transition probabilities.

The base case is deliberately cautious:

- HD and PD mortality are equal in the main model.
- The observational PD mortality hazard ratio is retained for scenario analysis only.
- PD-to-HD switching is included as technique failure/modality switch.
- Transplant is excluded from the base model.

In [11]:
transitions = tables["transitions"].copy()

transitions["transition_input_type"] = np.where(
    transitions["include_in_base_case"].fillna(False),
    "base_case_transition_input",
    np.where(transitions["qualifier"].astype(str).str.contains("excluded|observational", case=False, regex=True), "scenario_or_context", "supporting_transition_input")
)

transitions["transition_caution"] = np.where(
    transitions["qualifier"].astype(str).str.contains("assumption", case=False),
    "Assumption-heavy input; must be tested in sensitivity analysis.",
    "Use according to source and notes."
)

write_processed(transitions, "transition_inputs_processed.csv")

transitions_model_ready = transitions.loc[
    transitions["include_in_base_case"].fillna(False),
    ["parameter", "from_state", "to_state", "base_value", "low_value", "high_value", "unit", "source_key", "source_title", "source_url", "notes"]
].reset_index(drop=True)

write_processed(transitions_model_ready, "transitions_model_ready.csv")
transitions_model_ready

Wrote transition_inputs_processed.csv: 6 rows
Wrote transitions_model_ready.csv: 4 rows


,parameter,from_state,to_state,base_value,low_value,high_value,unit,source_key,source_title,source_url,notes
0,annual_death_probability_hd,HD,Death,0.12,0.08,0.18,annual_probability,KHOO_2022_OUTCOMES,Khoo CY et al. Death and cardiovascular outcom...,https://annals.edu.sg/death-and-cardiovascular...,Base case assumes equal annual mortality for H...
1,annual_death_probability_pd,PD,Death,0.12,0.08,0.20,annual_probability,KHOO_2022_OUTCOMES,Khoo CY et al. Death and cardiovascular outcom...,https://annals.edu.sg/death-and-cardiovascular...,Base case assumes equal mortality. Scenario an...
2,annual_pd_to_hd_switch_probability,PD,Switched_PD_to_HD,0.15,0.08,0.25,annual_probability,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Represents PD technique failure or modality sw...
3,annual_hd_to_pd_switch_probability,HD,PD,0.01,0.00,0.03,annual_probability,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Small switch probability because the policy qu...


## 10. Build base-case Markov transition matrix

Health states:

1. `HD`
2. `PD`
3. `Switched_PD_to_HD`
4. `Death`

Base-case logic:

- Patients on HD can remain on HD, switch to PD, or die.
- Patients on PD can remain on PD, switch to HD after PD technique failure, or die.
- Patients in `Switched_PD_to_HD` are treated as being on HD after switching, but retained as a separate state so the model can track technique failure.
- Death is absorbing.

This is still a simplified model. It does not yet include transplant, peritonitis, cardiovascular hospitalisation or separate APD/CAPD states.

In [12]:
# Extract transition probabilities.
p_death_hd = get_param(transitions, "annual_death_probability_hd")
p_death_pd = get_param(transitions, "annual_death_probability_pd")
p_pd_to_hd = get_param(transitions, "annual_pd_to_hd_switch_probability")
p_hd_to_pd = get_param(transitions, "annual_hd_to_pd_switch_probability")

states = ["HD", "PD", "Switched_PD_to_HD", "Death"]

# Base transition matrix. Rows must sum to 1.
transition_matrix = pd.DataFrame(0.0, index=states, columns=states)

transition_matrix.loc["HD", "PD"] = p_hd_to_pd
transition_matrix.loc["HD", "Death"] = p_death_hd
transition_matrix.loc["HD", "HD"] = 1 - p_hd_to_pd - p_death_hd

transition_matrix.loc["PD", "Switched_PD_to_HD"] = p_pd_to_hd
transition_matrix.loc["PD", "Death"] = p_death_pd
transition_matrix.loc["PD", "PD"] = 1 - p_pd_to_hd - p_death_pd

transition_matrix.loc["Switched_PD_to_HD", "Death"] = p_death_hd
transition_matrix.loc["Switched_PD_to_HD", "Switched_PD_to_HD"] = 1 - p_death_hd

transition_matrix.loc["Death", "Death"] = 1.0

# Validate transition matrix.
row_sums = transition_matrix.sum(axis=1)
negative_values = (transition_matrix < 0).any().any()

if np.allclose(row_sums.values, 1.0, atol=0.0001) and not negative_values:
    add_validation(validation, "Transition matrix", "PASS", "All rows sum to 1 and no negative probabilities found.", "calculation")
else:
    add_validation(validation, "Transition matrix", "FAIL", f"Row sums={row_sums.to_dict()}, negative_values={negative_values}", "critical")

transition_matrix_out = transition_matrix.reset_index().rename(columns={"index": "from_state"})
write_processed(transition_matrix_out, "transition_matrix_base_case.csv")
transition_matrix

Wrote transition_matrix_base_case.csv: 4 rows


,HD,PD,Switched_PD_to_HD,Death
HD,0.87,0.01,0.00,0.12
PD,0.00,0.73,0.15,0.12
Switched_PD_to_HD,0.00,0.00,0.88,0.12
Death,0.00,0.00,0.00,1.00


## 11. Process population and uptake scenarios

This creates the main budget impact scenario table.

The budget impact scenario table calculates new starts by modality for each year and scenario:

- `pd_new_starts = eligible patients × PD uptake`
- `hd_new_starts = eligible patients × HD uptake`

This table does not apply annual Markov transitions yet. That happens in the modelling notebook.

In [13]:
population = tables["population"].copy()
write_processed(population, "population_inputs_processed.csv")

modality_mix = tables["modality_mix"].copy()
write_processed(modality_mix, "modality_mix_processed.csv")

uptake = tables["uptake"].copy()
uptake["pd_new_starts"] = uptake["eligible_patients_per_year"] * uptake["pd_uptake"]
uptake["hd_new_starts"] = uptake["eligible_patients_per_year"] * uptake["hd_uptake"]
uptake["incremental_pd_uptake_vs_current"] = uptake["pd_uptake"] - uptake.groupby("year")["pd_uptake"].transform(
    lambda s: s[uptake.loc[s.index, "scenario"].eq("current_practice")].iloc[0] if uptake.loc[s.index, "scenario"].eq("current_practice").any() else np.nan
)
uptake["incremental_pd_new_starts_vs_current"] = uptake["eligible_patients_per_year"] * uptake["incremental_pd_uptake_vs_current"]

# Round patient counts for reporting but keep exact values in model fields.
uptake["pd_new_starts_rounded"] = uptake["pd_new_starts"].round(0).astype(int)
uptake["hd_new_starts_rounded"] = uptake["hd_new_starts"].round(0).astype(int)
uptake["incremental_pd_new_starts_rounded"] = uptake["incremental_pd_new_starts_vs_current"].round(0).astype(int)

write_processed(uptake, "uptake_scenarios_processed.csv")
write_processed(uptake, "budget_impact_scenarios_model_ready.csv")

uptake.head(12)

Wrote population_inputs_processed.csv: 5 rows
Wrote modality_mix_processed.csv: 6 rows
Wrote uptake_scenarios_processed.csv: 20 rows
Wrote budget_impact_scenarios_model_ready.csv: 20 rows


,year,scenario,incident_patients_per_year,pd_suitable_share,eligible_patients_per_year,pd_uptake,hd_uptake,notes,source_key,source_title,source_url,data_origin,pd_new_starts,hd_new_starts,incremental_pd_uptake_vs_current,incremental_pd_new_starts_vs_current,pd_new_starts_rounded,hd_new_starts_rounded,incremental_pd_new_starts_rounded
0,2026.0,current_practice,2190.0,0.6,1314.0,0.19,0.81,Counterfactual scenario keeps PD uptake consta...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,249.66,1064.34,0.00,0.00,250,1064,0
1,2026.0,low_pd_adoption,2190.0,0.6,1314.0,0.21,0.79,Conservative increase in PD uptake below the 3...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,275.94,1038.06,0.02,26.28,276,1038,26
2,2026.0,base_pd_adoption,2190.0,0.6,1314.0,0.22,0.78,Base scenario reaches the 30% PD uptake target...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,289.08,1024.92,0.03,39.42,289,1025,39
3,2026.0,high_pd_adoption,2190.0,0.6,1314.0,0.25,0.75,"Ambitious scenario exceeds the 30% target, use...",MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,328.50,985.50,0.06,78.84,328,986,79
4,2027.0,current_practice,2190.0,0.6,1314.0,0.19,0.81,Counterfactual scenario keeps PD uptake consta...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,249.66,1064.34,0.00,0.00,250,1064,0
5,2027.0,low_pd_adoption,2190.0,0.6,1314.0,0.23,0.77,Conservative increase in PD uptake below the 3...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,302.22,1011.78,0.04,52.56,302,1012,53
6,2027.0,base_pd_adoption,2190.0,0.6,1314.0,0.24,0.76,Base scenario reaches the 30% PD uptake target...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,315.36,998.64,0.05,65.70,315,999,66
7,2027.0,high_pd_adoption,2190.0,0.6,1314.0,0.30,0.70,"Ambitious scenario exceeds the 30% target, use...",MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,394.20,919.80,0.11,144.54,394,920,145
8,2028.0,current_practice,2190.0,0.6,1314.0,0.19,0.81,Counterfactual scenario keeps PD uptake consta...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,249.66,1064.34,0.00,0.00,250,1064,0
9,2028.0,low_pd_adoption,2190.0,0.6,1314.0,0.25,0.75,Conservative increase in PD uptake below the 3...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,328.50,985.50,0.06,78.84,328,986,79


## 12. Create base-case parameter table

The modelling notebook should be able to read one clean parameter file rather than repeatedly searching across raw CSVs.

This table contains the main parameters required for the first Markov and budget impact model.

In [14]:
def one_row_parameter(parameter, category, base, low, high, unit, source_key, source_title, source_url, notes):
    return {
        "parameter": parameter,
        "category": category,
        "base_value": base,
        "low_value": low,
        "high_value": high,
        "unit": unit,
        "source_key": source_key,
        "source_title": source_title,
        "source_url": source_url,
        "notes": notes
    }

parameter_rows = []

# Population and uptake parameters.
incident_row = population.loc[population["parameter"] == "estimated_new_kidney_failure_patients_per_year"].iloc[0]
parameter_rows.append(one_row_parameter(
    "incident_kidney_failure_patients_per_year", "population",
    incident_row["base_value"], incident_row["low_value"], incident_row["high_value"], incident_row["unit"],
    incident_row["source_key"], incident_row["source_title"], incident_row["source_url"],
    "Annual incident kidney failure patients derived from six new patients per day."
))

eligible_base = uptake.loc[uptake["scenario"] == "current_practice", "eligible_patients_per_year"].iloc[0]
pd_suitable_share = uptake.loc[uptake["scenario"] == "current_practice", "pd_suitable_share"].iloc[0]
parameter_rows.append(one_row_parameter(
    "pd_suitable_share", "population",
    pd_suitable_share, 0.40, 0.80, "proportion",
    "MOH_PD_POLICY", "Ministry of Health Singapore. Key causes of recent increase in number of patients requiring kidney dialysis",
    "https://www.moh.gov.sg/newsroom/key-causes-of-recent-increase-in-number-of-patients-requiring-kidney-dialysis/",
    "Assumption applied to incident patients to estimate medically suitable PD-eligible cohort."
))
parameter_rows.append(one_row_parameter(
    "eligible_patients_per_year", "population",
    eligible_base, np.nan, np.nan, "patients_per_year",
    "MOH_PD_POLICY", "Ministry of Health Singapore. Key causes of recent increase in number of patients requiring kidney dialysis",
    "https://www.moh.gov.sg/newsroom/key-causes-of-recent-increase-in-number-of-patients-requiring-kidney-dialysis/",
    "Derived as incident patients per year multiplied by PD suitability share."
))

current_pd = get_param(modality_mix, "current_pd_uptake_new_dialysis_patients")
target_pd = get_param(modality_mix, "target_pd_uptake_new_dialysis_patients")
parameter_rows.append(one_row_parameter(
    "current_pd_uptake", "uptake",
    current_pd, np.nan, np.nan, "proportion",
    "MOH_PD_POLICY", "Ministry of Health Singapore. Key causes of recent increase in number of patients requiring kidney dialysis",
    "https://www.moh.gov.sg/newsroom/key-causes-of-recent-increase-in-number-of-patients-requiring-kidney-dialysis/",
    "Current practice PD uptake anchor among new dialysis patients."
))
parameter_rows.append(one_row_parameter(
    "target_pd_uptake", "uptake",
    target_pd, np.nan, np.nan, "proportion",
    "MOH_PD_POLICY", "Ministry of Health Singapore. Key causes of recent increase in number of patients requiring kidney dialysis",
    "https://www.moh.gov.sg/newsroom/key-causes-of-recent-increase-in-number-of-patients-requiring-kidney-dialysis/",
    "Policy target PD uptake among new dialysis patients."
))

# Annual costs.
for _, row in annual_costs_model_ready.iterrows():
    parameter_rows.append(one_row_parameter(
        row["parameter"], "cost",
        row["base_value"], row["low_value"], row["high_value"], row["unit"],
        row["source_key"], row["source_title"], row["source_url"], row["notes"]
    ))

# Utilities.
for _, row in utilities_model_ready.iterrows():
    parameter_rows.append(one_row_parameter(
        row["parameter"], "utility",
        row["base_value"], row["low_value"], row["high_value"], row["unit"],
        row["source_key"], row["source_title"], row["source_url"], row["notes"]
    ))

# Transitions.
for _, row in transitions_model_ready.iterrows():
    parameter_rows.append(one_row_parameter(
        row["parameter"], "transition",
        row["base_value"], row["low_value"], row["high_value"], row["unit"],
        row["source_key"], row["source_title"], row["source_url"], row["notes"]
    ))

# Model constants.
parameter_rows.append(one_row_parameter(
    "time_horizon_years", "model_setting",
    5, 1, 10, "years",
    "SG_MTE_METHODS", "Singapore Medical Technology Evaluation Methods and Process Guide",
    "https://www.ace-hta.gov.sg/resources/process-methods/",
    "Base-case model and budget impact horizon."
))
parameter_rows.append(one_row_parameter(
    "cycle_length_years", "model_setting",
    1, 0.5, 1, "years",
    "DRUMMOND_2015", "Drummond MF et al. Methods for the Economic Evaluation of Health Care Programmes. 4th ed.",
    "NA",
    "Annual cycle length for first-pass cohort Markov model."
))

base_case_parameters = pd.DataFrame(parameter_rows)
write_processed(base_case_parameters, "base_case_parameters_model_ready.csv")
base_case_parameters

Wrote base_case_parameters_model_ready.csv: 19 rows


,parameter,category,base_value,low_value,high_value,unit,source_key,source_title,source_url,notes
0,incident_kidney_failure_patients_per_year,population,2190.00,NaN,NaN,patients_per_year,NKF_KEY_STATS,National Kidney Foundation Singapore. Key Stat...,https://nkfs.org/about-us/key-statistics/,Annual incident kidney failure patients derive...
1,pd_suitable_share,population,0.60,0.40,0.80,proportion,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Assumption applied to incident patients to est...
2,eligible_patients_per_year,population,1314.00,NaN,NaN,patients_per_year,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Derived as incident patients per year multipli...
3,current_pd_uptake,uptake,0.19,NaN,NaN,proportion,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Current practice PD uptake anchor among new di...
4,target_pd_uptake,uptake,0.30,NaN,NaN,proportion,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Policy target PD uptake among new dialysis pat...
5,annual_cost_hd,cost,37800.00,33600.00,42000.00,SGD_per_year,DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Annual HD cost used in base Markov model.
6,annual_cost_pd,cost,18840.00,16800.00,20700.00,SGD_per_year,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Annual blended PD cost used in base Markov model.
7,annual_cost_switched_pd_to_hd,cost,37800.00,33600.00,42000.00,SGD_per_year,DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Assumed equal to annual HD cost after PD techn...
8,annual_cost_death,cost,0.00,0.00,0.00,SGD_per_year,DRUMMOND_2015,Drummond MF et al. Methods for the Economic Ev...,NA,No ongoing dialysis cost assigned after death ...
9,health_state_utility_hd,utility,0.58,0.44,0.71,utility_weight,COOPER_2020_UTILITIES,Cooper JT et al. Health related quality of lif...,https://pmc.ncbi.nlm.nih.gov/articles/PMC7507735/,Placeholder utility for first-pass model. Coop...


## 13. Create state value table

The modelling notebook will join this table to each Markov state to calculate costs and QALYs.

In [16]:
state_values = (
    annual_costs_model_ready[["health_state", "base_value", "low_value", "high_value"]]
    .rename(columns={
        "base_value": "annual_cost_base",
        "low_value": "annual_cost_low",
        "high_value": "annual_cost_high"
    })
    .merge(
        utilities_model_ready[["health_state", "base_value", "low_value", "high_value"]].rename(columns={
            "base_value": "utility_base",
            "low_value": "utility_low",
            "high_value": "utility_high"
        }),
        on="health_state",
        how="outer"
    )
)

state_values["annual_cost_base"] = state_values["annual_cost_base"].fillna(0)
state_values["annual_cost_low"] = state_values["annual_cost_low"].fillna(0)
state_values["annual_cost_high"] = state_values["annual_cost_high"].fillna(0)
state_values["utility_base"] = state_values["utility_base"].fillna(0)
state_values["utility_low"] = state_values["utility_low"].fillna(0)
state_values["utility_high"] = state_values["utility_high"].fillna(0)

state_values = state_values[[
    "health_state",
    "annual_cost_base", "annual_cost_low", "annual_cost_high",
    "utility_base", "utility_low", "utility_high"
]]

write_processed(state_values, "state_values_model_ready.csv")
state_values

Wrote state_values_model_ready.csv: 4 rows


,health_state,annual_cost_base,annual_cost_low,annual_cost_high,utility_base,utility_low,utility_high
0,Death,0.0,0.0,0.0,0.00,0.00,0.00
1,HD,37800.0,33600.0,42000.0,0.58,0.44,0.71
2,PD,18840.0,16800.0,20700.0,0.62,0.53,0.72
3,Switched_PD_to_HD,37800.0,33600.0,42000.0,0.56,0.44,0.71


## 14. Create quality flags for report interpretation

This table is useful when writing limitations. It separates strong Singapore inputs from weaker assumption-based inputs.

In [17]:
quality_rows = []

def classify_quality(row):
    qualifier = str(row.get("qualifier", "")).lower()
    notes = str(row.get("notes", "")).lower()
    source_key = str(row.get("source_key", ""))

    if "assumption" in qualifier or "placeholder" in qualifier or "assumed" in notes:
        return "High uncertainty / assumption"
    if "derived" in qualifier or "weighted" in qualifier:
        return "Derived input"
    if source_key in ["MOH_PD_POLICY", "NKF_KEY_STATS", "DUKE_MYKIDNEY_COSTS", "HEALTHHUB_PD_COSTS"]:
        return "Singapore public source"
    if source_key.startswith("YANG") or source_key.startswith("KHOO") or source_key.startswith("KHAN"):
        return "Singapore peer-reviewed or sector evidence"
    if source_key in ["COOPER_2020_UTILITIES"]:
        return "International literature"
    return "Other"

for table_name in ["population", "modality_mix", "costs", "utilities", "transitions"]:
    df = tables[table_name].copy()
    if "parameter" not in df.columns:
        continue
    for _, row in df.iterrows():
        quality_rows.append({
            "table": table_name,
            "parameter": row.get("parameter"),
            "base_value": row.get("base_value"),
            "low_value": row.get("low_value"),
            "high_value": row.get("high_value"),
            "unit": row.get("unit"),
            "source_key": row.get("source_key"),
            "input_quality_flag": classify_quality(row),
            "notes": row.get("notes")
        })

input_quality_flags = pd.DataFrame(quality_rows)
write_processed(input_quality_flags, "input_quality_flags.csv")

input_quality_flags.groupby("input_quality_flag").size().reset_index(name="n_inputs")

Wrote input_quality_flags.csv: 35 rows


,input_quality_flag,n_inputs
0,Derived input,8
1,High uncertainty / assumption,7
2,Other,1
3,Singapore peer-reviewed or sector evidence,10
4,Singapore public source,9


## 15. Final validation and stop-if-critical-fail

This notebook should not quietly proceed if there is a critical failure.

In [18]:
validation_df = pd.DataFrame(validation).drop_duplicates(subset=["test", "status", "details", "severity"], keep="last")
write_processed(validation_df, "data_validation_summary.csv")

critical_fails = validation_df.loc[
    (validation_df["severity"] == "critical") & (validation_df["status"] == "FAIL")
]

print("Validation summary:")
display(validation_df.groupby(["severity", "status"]).size().reset_index(name="n"))

if not critical_fails.empty:
    print("\nCritical failures found:")
    display(critical_fails)
    raise ValueError("Critical validation failures found. Fix data_raw before running the modelling notebook.")
else:
    print("\nNo critical validation failures. data_processed is ready for the modelling notebook.")

Wrote data_validation_summary.csv: 24 rows
Validation summary:


,severity,status,n
0,calculation,PASS,10
1,critical,PASS,9
2,data_quality,PASS,5



No critical validation failures. data_processed is ready for the modelling notebook.


## 16. Processed data dictionary

This saves a simple dictionary of the generated processed files. It makes the workflow easier to explain in the final report.

In [19]:
processed_file_descriptions = [
    ("model_scope_processed.csv", "Clean report-facing scope and decision-problem table."),
    ("clinical_evidence_processed.csv", "Clean clinical and economic evidence summary table."),
    ("population_inputs_processed.csv", "Clean population inputs for burden and eligible population derivation."),
    ("modality_mix_processed.csv", "Clean current and target modality mix inputs."),
    ("cost_inputs_processed.csv", "Processed cost inputs with cost-quality caution flags."),
    ("annual_costs_model_ready.csv", "Annual cost by Markov health state."),
    ("utility_inputs_processed.csv", "Processed utility inputs with uncertainty labels."),
    ("utilities_model_ready.csv", "Utility weights by Markov health state."),
    ("transition_inputs_processed.csv", "Processed transition inputs with uncertainty labels."),
    ("transitions_model_ready.csv", "Base-case transition probabilities."),
    ("transition_matrix_base_case.csv", "Base-case Markov transition matrix."),
    ("uptake_scenarios_processed.csv", "Processed uptake scenario table with new-start calculations."),
    ("budget_impact_scenarios_model_ready.csv", "Budget impact scenario table for the modelling notebook."),
    ("base_case_parameters_model_ready.csv", "Single model-ready parameter table for base-case model."),
    ("state_values_model_ready.csv", "Cost and utility values by health state."),
    ("source_inventory.csv", "Inventory of sources used across processed raw data."),
    ("input_quality_flags.csv", "Flags showing whether inputs are Singapore source, derived, assumption-heavy or literature-based."),
    ("data_validation_summary.csv", "Validation test results for the processed dataset."),
]

processed_dictionary = pd.DataFrame(processed_file_descriptions, columns=["processed_file", "description"])
write_processed(processed_dictionary, "processed_data_dictionary.csv")
processed_dictionary

Wrote processed_data_dictionary.csv: 18 rows


,processed_file,description
0,model_scope_processed.csv,Clean report-facing scope and decision-problem...
1,clinical_evidence_processed.csv,Clean clinical and economic evidence summary t...
2,population_inputs_processed.csv,Clean population inputs for burden and eligibl...
3,modality_mix_processed.csv,Clean current and target modality mix inputs.
4,cost_inputs_processed.csv,Processed cost inputs with cost-quality cautio...
5,annual_costs_model_ready.csv,Annual cost by Markov health state.
6,utility_inputs_processed.csv,Processed utility inputs with uncertainty labels.
7,utilities_model_ready.csv,Utility weights by Markov health state.
8,transition_inputs_processed.csv,Processed transition inputs with uncertainty l...
9,transitions_model_ready.csv,Base-case transition probabilities.


## 17. Preview key model-ready files

These are the files that matter most for the next notebook.

In [20]:
print("Base-case parameters")
display(base_case_parameters)

print("\nState values")
display(state_values)

print("\nTransition matrix")
display(transition_matrix)

print("\nBudget impact scenario inputs")
display(uptake.head(12))

print("\nGenerated files in data_processed:")
for p in sorted(DATA_PROCESSED.glob("*.csv")):
    print("-", p.name)

Base-case parameters


,parameter,category,base_value,low_value,high_value,unit,source_key,source_title,source_url,notes
0,incident_kidney_failure_patients_per_year,population,2190.00,NaN,NaN,patients_per_year,NKF_KEY_STATS,National Kidney Foundation Singapore. Key Stat...,https://nkfs.org/about-us/key-statistics/,Annual incident kidney failure patients derive...
1,pd_suitable_share,population,0.60,0.40,0.80,proportion,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Assumption applied to incident patients to est...
2,eligible_patients_per_year,population,1314.00,NaN,NaN,patients_per_year,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Derived as incident patients per year multipli...
3,current_pd_uptake,uptake,0.19,NaN,NaN,proportion,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Current practice PD uptake anchor among new di...
4,target_pd_uptake,uptake,0.30,NaN,NaN,proportion,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Policy target PD uptake among new dialysis pat...
5,annual_cost_hd,cost,37800.00,33600.00,42000.00,SGD_per_year,DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Annual HD cost used in base Markov model.
6,annual_cost_pd,cost,18840.00,16800.00,20700.00,SGD_per_year,KHAN_2022_ECONOMICS,Khan BA et al. Health economics of kidney repl...,https://annals.edu.sg/health-economics-of-kidn...,Annual blended PD cost used in base Markov model.
7,annual_cost_switched_pd_to_hd,cost,37800.00,33600.00,42000.00,SGD_per_year,DUKE_MYKIDNEY_COSTS,Duke-NUS Lien Centre for Palliative Care. myKI...,https://www.duke-nus.edu.sg/lcpc/mykidney/trea...,Assumed equal to annual HD cost after PD techn...
8,annual_cost_death,cost,0.00,0.00,0.00,SGD_per_year,DRUMMOND_2015,Drummond MF et al. Methods for the Economic Ev...,NA,No ongoing dialysis cost assigned after death ...
9,health_state_utility_hd,utility,0.58,0.44,0.71,utility_weight,COOPER_2020_UTILITIES,Cooper JT et al. Health related quality of lif...,https://pmc.ncbi.nlm.nih.gov/articles/PMC7507735/,Placeholder utility for first-pass model. Coop...



State values


,health_state,annual_cost_base,annual_cost_low,annual_cost_high,utility_base,utility_low,utility_high
0,Death,0.0,0.0,0.0,0.00,0.00,0.00
1,HD,37800.0,33600.0,42000.0,0.58,0.44,0.71
2,PD,18840.0,16800.0,20700.0,0.62,0.53,0.72
3,Switched_PD_to_HD,37800.0,33600.0,42000.0,0.56,0.44,0.71



Transition matrix


,HD,PD,Switched_PD_to_HD,Death
HD,0.87,0.01,0.00,0.12
PD,0.00,0.73,0.15,0.12
Switched_PD_to_HD,0.00,0.00,0.88,0.12
Death,0.00,0.00,0.00,1.00



Budget impact scenario inputs


,year,scenario,incident_patients_per_year,pd_suitable_share,eligible_patients_per_year,pd_uptake,hd_uptake,notes,source_key,source_title,source_url,data_origin,pd_new_starts,hd_new_starts,incremental_pd_uptake_vs_current,incremental_pd_new_starts_vs_current,pd_new_starts_rounded,hd_new_starts_rounded,incremental_pd_new_starts_rounded
0,2026.0,current_practice,2190.0,0.6,1314.0,0.19,0.81,Counterfactual scenario keeps PD uptake consta...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,249.66,1064.34,0.00,0.00,250,1064,0
1,2026.0,low_pd_adoption,2190.0,0.6,1314.0,0.21,0.79,Conservative increase in PD uptake below the 3...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,275.94,1038.06,0.02,26.28,276,1038,26
2,2026.0,base_pd_adoption,2190.0,0.6,1314.0,0.22,0.78,Base scenario reaches the 30% PD uptake target...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,289.08,1024.92,0.03,39.42,289,1025,39
3,2026.0,high_pd_adoption,2190.0,0.6,1314.0,0.25,0.75,"Ambitious scenario exceeds the 30% target, use...",MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,328.50,985.50,0.06,78.84,328,986,79
4,2027.0,current_practice,2190.0,0.6,1314.0,0.19,0.81,Counterfactual scenario keeps PD uptake consta...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,249.66,1064.34,0.00,0.00,250,1064,0
5,2027.0,low_pd_adoption,2190.0,0.6,1314.0,0.23,0.77,Conservative increase in PD uptake below the 3...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,302.22,1011.78,0.04,52.56,302,1012,53
6,2027.0,base_pd_adoption,2190.0,0.6,1314.0,0.24,0.76,Base scenario reaches the 30% PD uptake target...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,315.36,998.64,0.05,65.70,315,999,66
7,2027.0,high_pd_adoption,2190.0,0.6,1314.0,0.30,0.70,"Ambitious scenario exceeds the 30% target, use...",MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,394.20,919.80,0.11,144.54,394,920,145
8,2028.0,current_practice,2190.0,0.6,1314.0,0.19,0.81,Counterfactual scenario keeps PD uptake consta...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,249.66,1064.34,0.00,0.00,250,1064,0
9,2028.0,low_pd_adoption,2190.0,0.6,1314.0,0.25,0.75,Conservative increase in PD uptake below the 3...,MOH_PD_POLICY,Ministry of Health Singapore. Key causes of re...,https://www.moh.gov.sg/newsroom/key-causes-of-...,Singapore policy statement on PD-preferred str...,328.50,985.50,0.06,78.84,328,986,79



Generated files in data_processed:
- annual_costs_model_ready.csv
- assumptions_cleaned.csv
- base_case_parameters_model_ready.csv
- budget_impact_scenarios_model_ready.csv
- clinical_evidence_cleaned.csv
- clinical_evidence_processed.csv
- cost_inputs_processed.csv
- costs_cleaned.csv
- data_validation_summary.csv
- input_quality_flags.csv
- modality_mix_cleaned.csv
- modality_mix_processed.csv
- model_scope_cleaned.csv
- model_scope_processed.csv
- population_cleaned.csv
- population_inputs_processed.csv
- processed_data_dictionary.csv
- source_inventory.csv
- state_values_model_ready.csv
- transition_inputs_processed.csv
- transition_matrix_base_case.csv
- transitions_cleaned.csv
- transitions_model_ready.csv
- uptake_cleaned.csv
- uptake_scenarios_processed.csv
- utilities_cleaned.csv
- utilities_model_ready.csv
- utility_inputs_processed.csv
